# Cuaderno deprecado — conservado como antecedente exploratorio

Este notebook se conserva como antecedente del proyecto. Su contribución
permitió validar el acceso, procesamiento diario y revisión inicial de
127 viviendas. El flujo canónico posterior (`01_limpieza_ideal.ipynb` y
`02_eda_ideal.ipynb`) amplía este trabajo con corrección horaria
(UTC → Europe/London), cobertura mensual, metadata oficial, unidad
hogar-mes, cinco variables del endpoint y pseudoetiquetas reproducibles.

**Este cuaderno no debe utilizarse para regenerar el dataset oficial.**
Para el dataset canónico usar `data/processed/ideal_monthly_features_labeled.parquet`.
Para análisis diario como vista descriptiva, puede usar
`data/processed/ideal_127_viviendas_diario_PRE_PARITY_DEPRECATED.parquet`.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os

for raiz, carpetas, archivos in os.walk("/content/drive/MyDrive"):
    for archivo in archivos:
        if "ideal_127" in archivo.lower():
            print(os.path.join(raiz, archivo))

/content/drive/MyDrive/Hackathon ONE G9/Data/ideal_127_viviendas_diario.parquet


In [4]:
import pandas as pd

ruta = "/content/drive/MyDrive/Hackathon ONE G9/Data/ideal_127_viviendas_diario.parquet"


df = pd.read_parquet(ruta)

df.head()

,home_id,fecha,potencia_promedio,potencia_maxima,horas_registradas,mediciones_totales,dia_completo
0,102,2017-03-09,771.788731,6428,14,45947,False
1,102,2017-03-10,523.829512,3374,24,81287,True
2,102,2017-03-11,451.380681,4245,24,84495,True
3,102,2017-03-12,909.852201,7431,24,84913,True
4,102,2017-03-13,636.947767,5645,24,80188,True


In [5]:
print("Filas y columnas:", df.shape)
print("Viviendas únicas:", df["home_id"].nunique())

df.info()

Filas y columnas: (33351, 7)
Viviendas únicas: 127
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33351 entries, 0 to 33350
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   home_id             33351 non-null  int64  
 1   fecha               33351 non-null  object 
 2   potencia_promedio   33351 non-null  float64
 3   potencia_maxima     33351 non-null  int64  
 4   horas_registradas   33351 non-null  int64  
 5   mediciones_totales  33351 non-null  int64  
 6   dia_completo        33351 non-null  bool   
dtypes: bool(1), float64(1), int64(4), object(1)
memory usage: 1.6+ MB


In [6]:
print("Duplicados:", df.duplicated().sum())

print("\nValores nulos:")
print(df.isna().sum())

print("\nDías completos e incompletos:")
print(df["dia_completo"].value_counts())

Duplicados: 0

Valores nulos:
home_id               0
fecha                 0
potencia_promedio     0
potencia_maxima       0
horas_registradas     0
mediciones_totales    0
dia_completo          0
dtype: int64

Días completos e incompletos:
dia_completo
True     25495
False     7856
Name: count, dtype: int64


In [7]:
# Cantidad de días según las horas registradas
distribucion_horas = (
    df["horas_registradas"]
    .value_counts()
    .sort_index()
)

distribucion_horas

,count
horas_registradas,
1,30
2,5
3,9
4,33
5,10
6,21
7,129
8,129
9,35


In [8]:
print(
    "Días con 24 horas:",
    (df["horas_registradas"] == 24).sum()
)

print(
    "Días con 20 a 23 horas:",
    df["horas_registradas"].between(20, 23).sum()
)

print(
    "Días con 12 a 19 horas:",
    df["horas_registradas"].between(12, 19).sum()
)

print(
    "Días con menos de 12 horas:",
    (df["horas_registradas"] < 12).sum()
)

Días con 24 horas: 25495
Días con 20 a 23 horas: 5237
Días con 12 a 19 horas: 2097
Días con menos de 12 horas: 522


In [9]:
df_limpio = df[
    df["horas_registradas"] >= 20
].copy()

print("Filas originales:", len(df))
print("Filas conservadas:", len(df_limpio))
print("Filas excluidas:", len(df) - len(df_limpio))
print("Viviendas conservadas:", df_limpio["home_id"].nunique())

Filas originales: 33351
Filas conservadas: 30732
Filas excluidas: 2619
Viviendas conservadas: 127


In [10]:
print("Tamaño del DataFrame limpio:", df_limpio.shape)
print("Viviendas únicas:", df_limpio["home_id"].nunique())
print("Horas mínimas registradas:", df_limpio["horas_registradas"].min())

Tamaño del DataFrame limpio: (30732, 7)
Viviendas únicas: 127
Horas mínimas registradas: 20


In [11]:
df_limpio["fecha"] = pd.to_datetime(
    df_limpio["fecha"],
    errors="coerce"
)

df_limpio.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30732 entries, 1 to 33350
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   home_id             30732 non-null  int64         
 1   fecha               30732 non-null  datetime64[ns]
 2   potencia_promedio   30732 non-null  float64       
 3   potencia_maxima     30732 non-null  int64         
 4   horas_registradas   30732 non-null  int64         
 5   mediciones_totales  30732 non-null  int64         
 6   dia_completo        30732 non-null  bool          
dtypes: bool(1), datetime64[ns](1), float64(1), int64(4)
memory usage: 1.7 MB


Se creó df_limpio conservando únicamente los días con al menos 20 horas registradas. De esta manera se evita comparar jornadas muy incompletas, cuyos valores podrían parecer artificialmente bajos, y se mantiene la mayor parte de la información disponible.

In [12]:
df_limpio[
    [
        "potencia_promedio",
        "potencia_maxima",
        "horas_registradas",
        "mediciones_totales"
    ]
].describe()

,potencia_promedio,potencia_maxima,horas_registradas,mediciones_totales
count,30732.000000,30732.000000,30732.000000,30732.000000
mean,388.748691,5945.116881,23.653098,78330.336783
std,230.075744,3040.266230,0.896170,11629.315060
min,18.861981,73.000000,20.000000,123.000000
25%,227.834473,3548.500000,24.000000,77281.750000
50%,343.201642,5062.000000,24.000000,82683.000000
75%,494.558698,8514.000000,24.000000,84920.250000
max,2262.110838,22640.000000,24.000000,86383.000000


In [14]:
import os

carpeta_salida = "/content/drive/MyDrive/Hackathon ONE G9/data"

os.makedirs(carpeta_salida, exist_ok=True)

ruta_salida = (
    carpeta_salida
    + "/ideal_127_viviendas_limpio.parquet"
)

df_limpio.to_parquet(
    ruta_salida,
    index=False
)

print("Archivo limpio guardado correctamente")
print(ruta_salida)

Archivo limpio guardado correctamente
/content/drive/MyDrive/Hackathon ONE G9/data/ideal_127_viviendas_limpio.parquet


In [15]:
ruta_csv = (
    carpeta_salida
    + "/ideal_127_viviendas_limpio.csv"
)

df_limpio.to_csv(
    ruta_csv,
    index=False
)

print("CSV limpio guardado correctamente")

CSV limpio guardado correctamente
